# MiniGPT-Stack DPO launcher (Phase 5) — THIN, all logic lives in train/

One Save Version = one MODE. Set MODE, run top to bottom, Save Version.

- `probe`  : measures real **T4** seconds/update and seconds/held-out-eval on a
  short run (~PROBE_STEPS). Writes `/kaggle/working/dpo_probe.json`. No full training.
- `sweep`  : the 3-lr sweep {1e-6, 5e-6, 2e-5} at beta 0.1, 2 epochs. A fresh
  `/tmp` ckpt_dir per arm. **Each arm raises on non-zero exit** (a crash is never
  swallowed) and the COMPLETE banner is suppressed unless every arm returned 0.
- `final`  : rerun/extend the winning LR into a persistent ckpt_dir, then run the
  generation-quality eval vs the SFT baseline.

- `baseline`: SFT-only generation-quality eval on the same 100 val prompts (no training; run it before any DPO arm).

Every arm uses the FULL 198-pair held-out split for best-checkpoint selection
(no subset; cheap on T4) and the by-prompt holdout split (no prompt leakage).

**Attach two datasets for every MODE:** `tinystories-shards-8192` (the proven
shards cell hard-asserts it; `final` also reads val-split prompts from it) and
the DPO init dataset carrying `sft_final_best.pt` (exactly 125,893,429 bytes)
and `pairs_graded_200_1500_dedup.jsonl` (1,300 records).


In [ ]:
MODE = "sweep"   # probe | sweep | final | baseline   <-- SET THIS, then run all

# ---- PINNED (Phase 5 plan) ----
BETA           = 0.1
LR_GRID        = [1e-6, 5e-6, 2e-5]
EPOCHS         = 2
BATCH          = 8
SEED           = 0
EVAL_EVERY     = 50       # steps between FULL held-out evals (198 pairs)
PATIENCE       = 3        # early-stop on held-out loss
HELDOUT_FRAC   = 0.1      # by PROMPT
MAX_HOURS      = 3.0      # clean save+exit before Kaggle kills the session
PROBE_STEPS    = 31       # steps 0,10,20,30 are logged: step 0 is cold, 10/20/30 steady
PROBE_N_PROMPTS = 20      # val prompts x 6 samples per model for the probe eot_rate
BEST_LR        = None     # <-- set from the sweep summary, then MODE="final"

WEIGHTS_FILE   = "sft_final_best.pt"
WEIGHTS_BYTES  = 125_893_429
PAIRS_FILE     = "pairs_graded_200_1500_dedup.jsonl"
PAIRS_BYTES    = 3_547_950
PAIRS_RECORDS  = 1300
# SHA-256 of the LOCAL files that were uploaded (checked inside the kernel)
WEIGHTS_SHA256 = "d043156ee2d23aed1336cb42c80efe8cd603c40421a469d4d9519d230cbc3a5c"
PAIRS_SHA256   = "596984d8f494bba986e050af00fa088adb1d3f30e5b1091ccf576fb769afbd39"

assert MODE in ("probe", "sweep", "final", "baseline"), MODE

import time
T0 = time.time()
HARD_CAP_S = {"probe": 20 * 60, "sweep": 60 * 60}.get(MODE)   # hard cap: probe 20 min, sweep 60 min


def time_left():
    """Seconds left under the hard cap, or None when no cap applies."""
    return None if HARD_CAP_S is None else HARD_CAP_S - (time.time() - T0)


def capped(argv):
    """subprocess.run with the remaining hard-cap budget as its timeout.
    TimeoutExpired propagates, which ends the kernel with an error."""
    import subprocess
    left = time_left()
    if left is not None and left < 30:
        raise RuntimeError(f"[dpo] hard cap: only {left:.0f}s left; not starting {argv[2:4]}")
    return subprocess.run(argv, timeout=left)


COMPLETED = False   # sentinel: set True ONLY by the mode's own success path;
                    # the final cell refuses to print COMPLETE while False.


OK_TERMINAL = ("max_steps", "epochs", "early_stop")   # a FINISHED arm; max_hours is not


def terminal_reason(ckpt_dir):
    """Why train.dpo stopped, from the terminal row it appends to its metrics.
    None if the run never wrote one (crash or old build)."""
    import json as _json, os as _os
    recs = [_json.loads(l) for l in open(_os.path.join(ckpt_dir, "metrics_dpo.jsonl"))
            if l.strip()]
    t = [r for r in recs if "terminal" in r]
    return t[-1]["terminal"] if t else None


def require_finished(tag, ckpt_dir):
    """Raise unless the arm ended for a real reason. rc==0 is not enough:
    a max_hours exit also returns 0 but the arm is INCOMPLETE."""
    why = terminal_reason(ckpt_dir)
    if why not in OK_TERMINAL:
        raise RuntimeError(f"[dpo] arm {tag} stopped with terminal={why!r}; "
                           f"expected one of {OK_TERMINAL}. NOT complete "
                           f"(rerun with --resume latest in a fresh session).")
    print(f"[dpo] arm {tag} terminal={why}", flush=True)


def log_run(tag, ckpt_dir, lr, steps=None):
    """Print the arm and every pinned flag as the FIRST lines of its log."""
    print("=" * 78)
    print(f"[dpo] RUN arm={tag} lr={lr:g} beta={BETA:g} ckpt_dir={ckpt_dir}")
    print(f"[dpo] flags epochs={EPOCHS} batch={BATCH} seed={SEED} "
          f"eval_every={EVAL_EVERY} patience={PATIENCE} "
          f"heldout_frac={HELDOUT_FRAC} max_hours={MAX_HOURS:g} steps={steps}")
    print("=" * 78, flush=True)


## GPU check (copied from kaggle_pretrain.ipynb)

In [ ]:
import torch
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (prep mode)")
print("torch", torch.__version__, "| cuda build", torch.version.cuda,
      "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable GPU in notebook settings"


## Clone (copied from kaggle_pretrain.ipynb)

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD

## Install + shard-independent tests (copied from kaggle_pretrain.ipynb)

Only the three suites that need no data shards run here. `test_phase4.py`
needs `data/meta.json`, so it runs in its own cell after the shards are linked.


In [ ]:
# Kaggle already ships numpy, pyyaml, tqdm, pytest, regex and datasets. Do NOT install the
# pinned requirements.txt here: it downgrades numpy and rebuilds it from source (~3 min).
!pip install -q regex pyyaml pytest tqdm datasets
!python -m pytest tests/test_config.py tests/test_model.py tests/test_pretrain.py -q 2>&1 | tail -n 2

## Inputs — DPO init (SFT weights + pairs) from a private dataset

Byte-exact on both files (a truncated upload or a wrong pairs file is fatal).
The SFT weights must load into the configured arch before any training.


In [ ]:
import glob, json, os, torch

def _find(name, nbytes):
    hits = glob.glob("/kaggle/input/**/" + name, recursive=True)
    assert hits, f"attach the private dataset containing {name}"
    got = os.path.getsize(hits[0])
    print(f"{name}: {hits[0]}")
    print(f"  bytes {got} (expected {nbytes})")
    assert got == nbytes, f"{name} size {got} != {nbytes} (truncated upload?)"
    return hits[0]

W = _find(WEIGHTS_FILE, WEIGHTS_BYTES)
PAIRS = _find(PAIRS_FILE, PAIRS_BYTES)
nrec = sum(1 for l in open(PAIRS, encoding="utf-8") if l.strip())
print("pairs records:", nrec, "(expected", PAIRS_RECORDS, ")")
assert nrec == PAIRS_RECORDS, f"pairs records {nrec} != {PAIRS_RECORDS}"

from common.config import load_config
from model.gpt import GPT, count_params
cfg = load_config("configs/base_30m.yaml")
_pm = GPT(cfg["model"])
_pm.load_state_dict(torch.load(W, map_location="cpu", weights_only=True))
print("SFT weights load into the configured arch:", True, "| params:", count_params(_pm))
del _pm


## Hash gate - both input files must be byte-identical to the local copies

SHA-256 is computed inside the kernel and compared with the hashes of the local files that were uploaded. A mismatch stops the run before any training.


In [ ]:
import hashlib

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

HASHES = {}
for _name, _path, _want in ((WEIGHTS_FILE, W, WEIGHTS_SHA256), (PAIRS_FILE, PAIRS, PAIRS_SHA256)):
    _got = _sha256(_path)
    HASHES[_name] = {"expected": _want, "got": _got, "match": _got == _want}
    print(f"[dpo] sha256 {_name}")
    print(f"[dpo]   expected {_want}")
    print(f"[dpo]   in-kernel {_got}  -> {'MATCH' if _got == _want else 'MISMATCH'}")
bad = [n for n, v in HASHES.items() if not v["match"]]
if bad:
    raise RuntimeError(f"[dpo] SHA-256 MISMATCH for {bad}; stopping before any training")
print("[dpo] hash gate: PASS (both files byte-identical to the local copies)", flush=True)


## Shards (copied from kaggle_pretrain.ipynb; prep branch removed)

This cell **hard-asserts** the shards dataset is attached, so attach **both**
datasets for every MODE (probe included) — it is kept byte-identical to the
proven SFT cell on purpose; the `final` eval reads val-split prompts from it.


In [ ]:
import glob, json, os
hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert hits, "attach the shards dataset first"
print("meta.json candidates:", hits)
shard_dir = os.path.dirname(hits[0])
print("--- meta.json head ---")
print("\n".join(open(os.path.join(shard_dir, "meta.json")).read().splitlines()[:15]))
os.makedirs("data", exist_ok=True)
# The cloned repo can contain a stale data/meta.json (smoke-test counts). If it is left in
# place, the symlink below is skipped and training would read the wrong token counts.
for f in glob.glob("data/*.bin") + ["data/meta.json"]:
    if os.path.lexists(f):
        os.remove(f)
for f in glob.glob(os.path.join(shard_dir, "*.bin")) + [os.path.join(shard_dir, "meta.json")]:
    os.symlink(f, os.path.join("data", os.path.basename(f)))
print("linked:", sorted(os.listdir("data")))
meta = json.load(open("data/meta.json"))
assert "shards" in meta and "vocab_size" in meta, meta.keys()
for s in meta["shards"]:
    assert {"split", "index", "file", "tokens"} <= set(s), s
print("meta keys OK:", sorted(meta.keys()))
tr = sum(s["tokens"] for s in meta["shards"] if s["split"] == "train")
va = sum(s["tokens"] for s in meta["shards"] if s["split"] == "validation")
print(f"EXACT counts from data/meta.json: train={tr} val={va} vocab={meta['vocab_size']}")
for s in meta["shards"]:
    nbytes = os.path.getsize(os.path.join("data", s["file"]))
    print(f"  {s['file']} {s['tokens']} toks {nbytes/1024/1024:.1f}MB")
    assert nbytes == 2 * s["tokens"], f"{s['file']}: file size does not match meta (stale meta.json?)"
assert tr > 400_000_000, f"train tokens {tr} look wrong (expected ~467M)"
TRAIN_TOKS, VAL_TOKS = tr, va

## Phase 4 tests — after the shards, and fatal on failure

These tests read `data/meta.json`, so they cannot run with the early suites.
They are run through `subprocess` so a non-zero exit raises: a `!pytest` magic
would only print the failure and let training continue on a broken tree.


In [ ]:
import subprocess, sys

proc = subprocess.run([sys.executable, "-m", "pytest",
                       "tests/test_phase4.py", "-q"])
if proc.returncode != 0:
    raise RuntimeError(f"[sft] tests/test_phase4.py FAILED "
                       f"(rc={proc.returncode}); stopping")
print("[sft] tests/test_phase4.py: PASS", flush=True)

## Probe - real T4 timing, GPU memory, step-0 sanity and eot_rate

A `PROBE_STEPS`-step run at lr 1e-6 (full held-out evals on). Step 0 is cold (CUDA warm-up); steps 10/20/30 are the steady `sec_per_update`. Peak GPU memory is sampled from `nvidia-smi` while the run is live. The step-0 held-out numbers come from the pre-update eval (expect loss ln2 = 0.6931, acc 0.00). Then `eval_quality` runs on `PROBE_N_PROMPTS` val prompts for the SFT model and the probe checkpoint (saved at step 0, i.e. after ONE update, so it is ~SFT: a real DPO eot_rate needs the sweep). Everything is under the hard 20-minute cap. Writes `/kaggle/working/dpo_probe.json`.


In [ ]:
import json, math, os, re, shutil, subprocess, threading, time, glob

def _gpu_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used",
                          "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.split()
    return int(out[0]) if out else 0

if MODE != "probe":
    print("skip: probe runs in probe mode only")
else:
    d = "/tmp/dpo_probe"
    shutil.rmtree(d, ignore_errors=True)
    log_run("probe", d, 1e-6, steps=PROBE_STEPS)
    argv = ["python", "-m", "train.dpo",
            "--config", "configs/base_30m.yaml", "--init_policy", W,
            "--pairs", PAIRS, "--ckpt_dir", d, "--beta", str(BETA),
            "--lr", str(1e-6), "--epochs", str(EPOCHS), "--batch", str(BATCH),
            "--eval_every", str(EVAL_EVERY), "--patience", str(PATIENCE),
            "--heldout_frac", str(HELDOUT_FRAC), "--seed", str(SEED),
            "--max_steps", str(PROBE_STEPS)]
    peak = {"mib": 0, "base": _gpu_mib()}
    stop = threading.Event()

    def _poll():
        while not stop.is_set():
            peak["mib"] = max(peak["mib"], _gpu_mib())
            time.sleep(0.5)

    th = threading.Thread(target=_poll, daemon=True)
    th.start()
    t0 = time.time()
    try:
        proc = capped(argv)
    finally:
        stop.set()
        th.join()
    wall = time.time() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"[dpo] PROBE exited {proc.returncode}; aborting before sweep")
    require_finished("probe", d)
    recs = [json.loads(l) for l in open(os.path.join(d, "metrics_dpo.jsonl")) if l.strip()]
    pre = next(r for r in recs if r.get("pre_update"))
    ts_all = {r["step"]: r["t_step"] for r in recs if "t_step" in r}
    steady = [v for k, v in sorted(ts_all.items()) if k >= 1]
    ev = [r for r in recs if "h_loss" in r]
    t_eval_warm = [r["t_eval"] for r in ev if "t_eval" in r]
    term = [r for r in recs if "terminal" in r][-1]
    step0_ok = abs(pre["h0_loss"] - math.log(2)) < 0.01 and pre["h0_acc"] <= 0.02
    info = {"gpu": torch.cuda.get_device_name(0),
            "sec_per_update": sum(steady) / len(steady),
            "sec_per_update_samples": ts_all,
            "sec_eval_cold": pre["t_eval"],
            "sec_eval": (sum(t_eval_warm) / len(t_eval_warm)) if t_eval_warm else None,
            "step0_h_loss": pre["h0_loss"], "step0_h_acc": pre["h0_acc"],
            "step0_ok": step0_ok,
            "updates_done": term["updates_done"],
            "peak_gpu_mib_total": peak["mib"], "gpu_mib_before_run": peak["base"],
            "peak_gpu_mib_run": peak["mib"] - peak["base"],
            "dpo_wall_s": wall, "hashes": HASHES}
    print(f"[dpo] measured {info['sec_per_update']:.3f} s/update (steady, steps "
          f"{sorted(k for k in ts_all if k >= 1)}), step0 {ts_all.get(0, float('nan')):.3f}s cold")
    print(f"[dpo] held-out eval: cold {info['sec_eval_cold']:.1f}s, warm {info['sec_eval']:.1f}s")
    print(f"[dpo] step-0 held-out loss={pre['h0_loss']:.4f} acc={pre['h0_acc']:.2f} "
          f"(expect 0.6931 / 0.00) -> {'OK' if step0_ok else 'UNEXPECTED'}")
    print(f"[dpo] peak GPU memory {peak['mib']} MiB total "
          f"({peak['mib'] - peak['base']} MiB above the notebook's own {peak['base']} MiB)")

    # eot_rate: SFT model vs the probe checkpoint (after one update, ~SFT)
    ws = sorted(glob.glob(os.path.join(d, "weights_step*.pt")),
                key=lambda x: int(re.search(r"weights_step(\d+)", x).group(1)))
    assert ws, f"no probe weights in {d}"
    info["probe_weights"] = os.path.basename(ws[-1])
    for label, wt in (("sft_baseline", W), ("dpo_probe", ws[-1])):
        out = f"/kaggle/working/probe_eval_{label}.jsonl"
        proc = capped(["python", "train/eval_quality.py",
                       "--config", "configs/base_30m.yaml",
                       "--weights", wt, "--ppl_weights", W, "--out", out,
                       "--n_prompts", str(PROBE_N_PROMPTS), "--n_samples", "6",
                       "--label", label])
        if proc.returncode != 0:
            raise RuntimeError(f"[dpo] probe eval {label} exited {proc.returncode}")
        s = json.load(open(out + ".summary.json"))
        info["eval_" + label] = s
        print(f"[dpo] {label:12} eot_rate={s['eot_rate']:.3f} cov3={s['cov3']:.3f} "
              f"full3={s['full3_rate']:.3f} words={s['mean_words']:.1f} "
              f"gold_ppl={s['gold_ppl']:.2f} ({s['seconds']:.0f}s)")
    info["kernel_elapsed_s"] = time.time() - T0
    json.dump(info, open("/kaggle/working/dpo_probe.json", "w"), indent=1)
    print(f"[dpo] kernel elapsed {info['kernel_elapsed_s']:.0f}s of the "
          f"{HARD_CAP_S}s hard cap")
    if not step0_ok:
        raise RuntimeError("[dpo] step-0 held-out numbers are not ~ln2/0.00; "
                           "NOT reporting COMPLETE (see dpo_probe.json)")
    COMPLETED = True        # hash gate + probe rc 0 + finished terminal + step-0 sane + evals ran


## Sweep — one fresh ckpt_dir per arm, guarded (raise on any failure)

Each arm calls `train.dpo` with `--resume latest` so a Kaggle preemption resumes
instead of restarting. `run_arm` returns the arm's summary; **a non-zero exit
raises immediately** and `COMPLETED` is never set, so the banner cell below stays
silent after any crash. Best-checkpoint selection uses the FULL held-out set.


In [ ]:
import glob, json, os, re, shutil, subprocess

TAGS = {}
ARM_SUMMARY = {}

def run_arm(tag, lr, steps=None, ckpt_dir=None, max_hours=MAX_HOURS):
    """Run one DPO arm. Fresh ckpt_dir unless given. RAISES on non-zero exit."""
    d = ckpt_dir or f"/tmp/dpo_{tag}"
    if ckpt_dir is None:
        shutil.rmtree(d, ignore_errors=True)          # FRESH dir per arm
    log_run(tag, d, lr, steps=steps)
    argv = ["python", "-m", "train.dpo",
            "--config", "configs/base_30m.yaml", "--init_policy", W,
            "--pairs", PAIRS, "--ckpt_dir", d, "--beta", str(BETA),
            "--lr", str(lr), "--epochs", str(EPOCHS), "--batch", str(BATCH),
            "--eval_every", str(EVAL_EVERY), "--patience", str(PATIENCE),
            "--heldout_frac", str(HELDOUT_FRAC), "--seed", str(SEED),
            "--resume", "latest", "--max_hours", str(max_hours)]
    if steps:
        argv += ["--max_steps", str(steps)]
    proc = capped(argv)          # hard cap: TimeoutExpired ends the kernel
    if proc.returncode != 0:
        raise RuntimeError(f"[dpo] arm {tag} FAILED (rc={proc.returncode}); "
                           f"NOT reporting COMPLETE")
    require_finished(tag, d)
    mf = os.path.join(d, "metrics_dpo.jsonl")
    recs = [json.loads(l) for l in open(mf) if l.strip()]
    h = [(r["step"], r["h_loss"], r["h_acc"]) for r in recs if "h_loss" in r]
    best = min(h, key=lambda t: t[1]) if h else None   # best held-out LOSS (train.dpo's own criterion)
    term = [r for r in recs if "terminal" in r][-1]
    return {"ckpt_dir": d, "terminal": term["terminal"],
            "updates_done": term["updates_done"],
            "best_step": best[0] if best else None,
            "best_h_loss": best[1] if best else None, "best_h_acc": best[2] if best else None}


if MODE != "sweep":
    print("skip: sweep runs in sweep mode only")
else:
    try:
        for lr in LR_GRID:
            tag = f"lr{lr:g}".replace(".", "p").replace("-", "m")
            TAGS[lr] = tag
            ARM_SUMMARY[lr] = run_arm(tag, lr)
    except Exception as e:
        print(f"[dpo] SWEEP ABORTED: {e}")
        raise                                   # never swallow a crash
    # copy each arm's best weights out so they survive the version
    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    for lr, tag in TAGS.items():
        d = f"/tmp/dpo_{tag}"
        ws = sorted(glob.glob(os.path.join(d, "weights_step*.pt")),
                    key=lambda x: int(re.search(r"weights_step(\d+)", x).group(1)))
        if ws:
            shutil.copy(ws[-1], f"/kaggle/working/checkpoints/dpo_{tag}_weights.pt")
    print("\n[dpo] === SWEEP SUMMARY (held-out LOSS; lower is better) ===")
    for lr in LR_GRID:
        s = ARM_SUMMARY.get(lr)
        if s:
            print(f"[dpo]   lr={lr:g}: h_loss={s['best_h_loss']:.4f} "
                  f"h_acc={s['best_h_acc']:.4f} @step {s['best_step']}")
    # ---- per-eval tables, divergence/NaN scan, artifacts for download ----
    import math as _m
    BAD, SWEEP_OUT = [], {}
    for lr, tag in TAGS.items():
        mf = f"/tmp/dpo_{tag}/metrics_dpo.jsonl"
        shutil.copy(mf, f"/kaggle/working/sweep_{tag}_metrics.jsonl")
        rows = [json.loads(l) for l in open(mf) if l.strip()]
        s = ARM_SUMMARY[lr]
        print(f"[dpo] --- per-eval held-out, lr={lr:g} ---")
        pre = [r for r in rows if r.get("pre_update")]
        if pre:
            print(f"[dpo]   pre-update   h_loss={pre[0]['h0_loss']:.4f} h_acc={pre[0]['h0_acc']:.3f}")
        evs = []
        for r in rows:
            if "h_loss" in r:
                evs.append({"step": r["step"], "h_loss": r["h_loss"], "h_acc": r["h_acc"],
                            "train_loss": r["loss"], "t_eval": r.get("t_eval")})
                print(f"[dpo]   step {r['step']:>4} h_loss={r['h_loss']:.4f} "
                      f"h_acc={r['h_acc']:.3f} train_loss={r['loss']:.4f}")
            for k in ("loss", "h_loss", "margin"):
                if k in r and not _m.isfinite(r[k]):
                    BAD.append((lr, r.get("step"), k))
        tsteps = [r["t_step"] for r in rows if "t_step" in r and r.get("step", 0) >= 1]
        diverged = any(e["h_loss"] > 0.75 or e["train_loss"] > 1.0 for e in evs)
        print(f"[dpo]   terminal={s['terminal']} updates_done={s['updates_done']} "
              f"best_step={s['best_step']} best_h_loss={s['best_h_loss']:.4f} "
              f"sec/update={sum(tsteps) / max(len(tsteps), 1):.3f} "
              f"diverged={'YES' if diverged else 'no'}")
        SWEEP_OUT[tag] = {"lr": lr, "evals": evs, "summary": s,
                          "sec_per_update": sum(tsteps) / max(len(tsteps), 1),
                          "diverged": diverged}
    json.dump(SWEEP_OUT, open("/kaggle/working/sweep_summary.json", "w"), indent=1)
    print("[dpo] eot_rate: NOT computed in sweep mode (final mode runs eval_quality)")
    print(f"[dpo] sweep kernel elapsed {time.time() - T0:.0f}s of the {HARD_CAP_S}s hard cap")
    if BAD:
        raise RuntimeError(f"[dpo] non-finite values in sweep arms: {BAD}")
    COMPLETED = True        # ONLY reached if every arm returned 0 and finished for a real reason
    print("[dpo] all arms returned 0; setting COMPLETED=True")


## Final — winning LR, persistent + resumable, then eval vs SFT baseline

Reruns the winning arm into a persistent `ckpt_dir` (resumes across Kaggle
sessions), then runs `train/eval_quality.py` on the DPO weights AND the SFT
baseline so the numbers sit side by side.


In [ ]:
import os, subprocess

if MODE != "final":
    print("skip: final runs in final mode only")
elif BEST_LR is None:
    print("BEST_LR is None: run MODE='sweep' and paste the winner into the flags cell.")
else:
    CKPT_DIR = "/kaggle/working/checkpoints/dpo_final"
    os.makedirs(CKPT_DIR, exist_ok=True)
    log_run("final", CKPT_DIR, BEST_LR)
    argv = ["python", "-m", "train.dpo",
            "--config", "configs/base_30m.yaml", "--init_policy", W,
            "--pairs", PAIRS, "--ckpt_dir", CKPT_DIR, "--beta", str(BETA),
            "--lr", str(BEST_LR), "--epochs", str(EPOCHS), "--batch", str(BATCH),
            "--eval_every", str(EVAL_EVERY), "--patience", str(PATIENCE),
            "--heldout_frac", str(HELDOUT_FRAC), "--seed", str(SEED),
            "--resume", "latest", "--max_hours", str(MAX_HOURS)]
    proc = subprocess.run(argv)
    if proc.returncode != 0:
        raise RuntimeError(f"[dpo] FINAL exited {proc.returncode}; NOT reporting COMPLETE")
    require_finished("final", CKPT_DIR)
    COMPLETED = True
    print("[dpo] final returned 0 with a finished terminal reason; "
          "setting COMPLETED=True")


## Generation-quality eval (final only) — DPO arm vs SFT baseline, side by side

100 val-split template prompts x n=6 samples (same sampler settings as the probe):
whole-word coverage, EOT rate, mean length, distinct-2, and perplexity under the
frozen SFT model. SFT baseline numbers printed beside the DPO arm's.


In [ ]:
import glob, json, os, re, subprocess

if MODE != "final":
    print("skip: eval runs in final mode only")
else:
    assert COMPLETED, "final arm did not complete; refusing to eval/report"
    dpo_w = "/kaggle/working/checkpoints/dpo_final"
    ws = sorted(glob.glob(os.path.join(dpo_w, "weights_step*.pt")),
                key=lambda x: int(re.search(r"weights_step(\d+)", x).group(1)))
    assert ws, f"no DPO weights in {dpo_w}"
    DPO_W = ws[-1]
    for label, wt in (("sft_baseline", W), ("dpo_arm", DPO_W)):
        out = f"/kaggle/working/eval_{label}.jsonl"
        argv = ["python", "train/eval_quality.py",
                "--config", "configs/base_30m.yaml",
                "--weights", wt, "--ppl_weights", W, "--out", out,
                "--n_prompts", "100", "--n_samples", "6", "--label", label]
        proc = subprocess.run(argv)
        if proc.returncode != 0:
            raise RuntimeError(f"[dpo] eval {label} exited {proc.returncode}")
    print("\n[dpo] === EVAL SUMMARY (SFT baseline | DPO arm) ===")
    for label in ("sft_baseline", "dpo_arm"):
        s = json.load(open(f"/kaggle/working/eval_{label}.jsonl.summary.json"))
        print(f"[dpo] {label:12} cov3={s['cov3']:.3f} full3={s['full3_rate']:.3f} "
              f"eot={s['eot_rate']:.3f} words={s['mean_words']:.1f} "
              f"dist2={s['distinct2']:.3f} ppl_sft={s['ppl_sft']:.2f} gold_ppl={s['gold_ppl']:.2f}")


## SFT baseline eval (baseline mode only)

Scores the SFT weights alone on the same 100 val prompts x 6 samples, so the baseline exists before any DPO arm and costs no training.


In [ ]:
import json, subprocess

if MODE != "baseline":
    print("skip: baseline runs in baseline mode only")
else:
    out = "/kaggle/working/eval_sft_baseline.jsonl"
    argv = ["python", "train/eval_quality.py",
            "--config", "configs/base_30m.yaml",
            "--weights", W, "--ppl_weights", W, "--out", out,
            "--n_prompts", "100", "--n_samples", "6", "--label", "sft_baseline"]
    proc = subprocess.run(argv)
    if proc.returncode != 0:
        raise RuntimeError(f"[dpo] baseline eval exited {proc.returncode}")
    s = json.load(open(out + ".summary.json"))
    print(f"[dpo] sft_baseline cov3={s['cov3']:.3f} full3={s['full3_rate']:.3f} "
          f"eot={s['eot_rate']:.3f} words={s['mean_words']:.1f} "
          f"dist2={s['distinct2']:.3f} ppl_sft={s['ppl_sft']:.2f} gold_ppl={s['gold_ppl']:.2f}")
    COMPLETED = True


## COMPLETE guard — never claims success after a crash

Prints the COMPLETE banner **only** if the COMPLETED sentinel is True (every
arm/eval returned 0). Otherwise it raises, so a crashed run can never be
mistaken for a finished one.


In [ ]:
if COMPLETED:
    print("=" * 78)
    print(f"[dpo] COMPLETE  mode={MODE}")
    print("=" * 78, flush=True)
else:
    raise RuntimeError("[dpo] COMPLETED sentinel is False -- a step was skipped "
                       "or crashed. NOT reporting COMPLETE.")
